In [ ]:
import pandas as pd
import numpy as np

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import matplotlib.pyplot as plt

In [17]:
df = pd.read_excel('bw_data.xlsx')

df["fwd_60d_return"] = np.log(df["XRT Price"].shift(-60) / df["XRT Price"])

raw_signals = (df['Deviation'] >= 2.0).astype(int)
signal = pd.Series(0, index=df.index)

i = 0
while i < len(df):
    if raw_signals.iloc[i] == 1:
        signal.iloc[i] = 1
        i += 30   # skip next 30 days
    else:
        i += 1

df["signal"] = signal
hit_df = df[df['signal'] == 1].copy()

In [75]:
def sharpe_matrix_calculations(
    df : pd.DataFrame,
    hit_df : pd.DataFrame
    ):
    """
    Create a matrix of stock path vectors (returns) after (and including) the signal
    - column is date of signal
    """

    # Price matrix per signal
    matrix_dict = {}
    for i in range(len(hit_df)):
        signal_date = hit_df['Date'].iloc[i]
        price_vector = df.loc[(df['Date'] >= signal_date)].iloc[:61]['XRT Price'].values
        matrix_dict[signal_date.strftime('%Y-%m-%d')] = price_vector
    raw_matrix_df = pd.DataFrame(matrix_dict)
    raw_matrix_df.index.name = "offset_days"

    # Daily return matrix per signal
    daily_return_df = np.log(raw_matrix_df.shift(1) / raw_matrix_df).dropna()

    # Sharpe value per signal (daily and annualized)
    sharpes = daily_return_df.mean() / daily_return_df.std()
    avg_sharpe = sharpes.mean()

    # For the 60-day hold, the daily and annual sharpe per signal are computed below.
    sharpe_df = (
        sharpes
        .to_frame()
        .rename(columns={0:'daily_sharpe'})
        .assign(
            annualized_sharpe = lambda x: x['daily_sharpe'] * np.sqrt(252)
        )
    )

    return sharpe_df


In [ ]:
sharpe_df = sharpe_matrix_calculations(df=df,hit_df=hit_df)

,daily_sharpe,annualized_sharpe
2008-05-15,0.048141,0.764213
2008-09-02,0.230929,3.665880
2009-04-03,-0.080119,-1.271852
2011-10-27,-0.026996,-0.428550
2013-05-14,-0.100703,-1.598617
2014-11-24,-0.072979,-1.158501
2016-03-01,0.072745,1.154786
2017-11-29,-0.033352,-0.529440
2018-01-12,0.068017,1.079742
2018-06-11,-0.059693,-0.947598
